# 05 — Sampling and Post-Processing

**Purpose:** Generate CIB–tSZ map pairs from a trained checkpoint and apply
post-sampling variance rescaling.

This notebook demonstrates the full generation workflow:

1. **Loading a checkpoint** — rebuilds the `Unet` + `GaussianDiffusion` model with the
   same architecture as training and loads weights from a `.pt` checkpoint via the
   `load_checkpoint` function in `sample.py`.

2. **Generating samples** — runs the reverse diffusion process to produce batches of
   `(N, 2, 256, 256)` CIB–tSZ patch pairs. Raw outputs are in the model's internal
   normalised range.

3. **Variance rescaling** — applies post-sampling correction to recover the true pixel
   intensity scale. The paper describes multiplying by a single scalar factor
   (σ_Agora / σ_DDPM: 1.0328 for CIB, 1.1425 for tSZ); the codebase implements this
   via `renormalize_dm_maps`, which applies a two-step affine transform. Both approaches
   are shown and compared. See `docs/paper_code_inconsistencies.md` for details.

**Inputs:**
- Trained checkpoint: `results/model-14.pt`
- Training maps (for rescaling statistics): `data/low_pass/2mJy/*.npy`

**Outputs:**
- Generated samples: `data/low_pass/2mJy/new_samples_cib_tsz_2mJy_lp.npy`

**Key module functions:**
- `foregrounds_diffusion.sample.build_model`
- `foregrounds_diffusion.sample.load_checkpoint`
- `foregrounds_diffusion.sample.sample`
- `foregrounds_diffusion.preprocessing.renormalize_dm_maps`

**Paper reference:** §3.2 (variance rescaling), §4 (generated sample evaluation).

In [ ]:
import numpy as np
import torch
from accelerate import Accelerator

from foregrounds_diffusion.sample import build_model, load_checkpoint, sample
from foregrounds_diffusion.preprocessing import renormalize_dm_maps

CHECKPOINT  = "results/model-14.pt"
OUTPUT_PATH = "data/low_pass/2mJy/new_samples_cib_tsz_2mJy_lp.npy"
N_BATCHES   = 5
BATCH_SIZE  = 16
PTSRC       = 2


In [ ]:
accelerator = Accelerator(split_batches=True, mixed_precision='fp16')
print(f"Device: {accelerator.device}")

diffusion = build_model(channels=2)
diffusion = diffusion.to(accelerator.device)
diffusion = load_checkpoint(diffusion, CHECKPOINT, accelerator)
print("Checkpoint loaded.")


In [ ]:
# Reverse diffusion: T steps of denoising from N(0,I) → map
all_samples = sample(diffusion, accelerator,
                     num_batches=N_BATCHES, batch_size=BATCH_SIZE)
print(f"Raw sample shape (channels-first): {all_samples.shape}")
print(f"Raw value range: [{all_samples.min():.4f}, {all_samples.max():.4f}]")


In [ ]:
# Load training maps for rescaling statistics
cib_train = np.load(f"data/low_pass/{PTSRC}mJy/CIB_map_150GHz_256_st6_minmax_{PTSRC}mJy_zero_lp.npy")
tsz_train = np.load(f"data/low_pass/{PTSRC}mJy/tSZ3_map_150GHz_256_st6_minmax_{PTSRC}mJy_norm_lp.npy")
train_maps = np.concatenate([cib_train, tsz_train], axis=-1)  # (N, H, W, 2)

# --- Code implementation: two-step affine transform ---
rescaled_code = renormalize_dm_maps(all_samples, train_maps, variance_scaling=True)
print(f"Rescaled (code) shape: {rescaled_code.shape}")

# --- Paper description: scalar multiply by sigma_Agora / sigma_DDPM ---
PAPER_SCALES = {'CIB': 1.0328, 'tSZ': 1.1425}  # from paper §3.2
rescaled_paper = all_samples.copy()
for c, (name, scale) in enumerate(PAPER_SCALES.items()):
    rescaled_paper[:, c] *= scale
    print(f"{name} scale factor (paper): {scale}")

# See docs/paper_code_inconsistencies.md for discussion of the difference


In [ ]:
from pathlib import Path
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_PATH, rescaled_code)
print(f"Saved {rescaled_code.shape[0]} samples → {OUTPUT_PATH}")
